In [2]:
from datetime import datetime
import glob
import shutil
import time

import win32com.client  # Windows COM 인터페이스 사용
import xml.etree.ElementTree as ET
import pandas as pd

import os.path

# 입력 파라미터
###################################################################################################################
network_path = r"C:\Users\alswl\Desktop\표준화 네트워크_260519\15km 시나리오\서비스 B"
network_file_name = r"연결로 우회분석_15km_260608.inpx"
network_full_path = os.path.join(network_path, network_file_name)

copy_network_path = os.path.join(network_path, f"연결로 우회분석 결과_서비스 B_15km_260608.inpx")

csv_path = r"C:\Users\alswl\Desktop\표준화 네트워크_260519\15km 시나리오\서비스 B"
csv_file_name = r"서비스 B.csv"

# 유고 발생 시각
acc_start_time = 1800
###################################################################################################################
# =========================================================
# .mer → parquet 변환 함수
# =========================================================
def convert_mer_to_parquet(mer_file, delete_original=True):
    try:
        print(f"[변환 시작] {mer_file}")

        with open(mer_file, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()

        data_start_idx = None
        for i, line in enumerate(lines):
            if "Measurem." in line:
                data_start_idx = i
                break

        if data_start_idx is None:
            print(f"[건너뜀] 데이터 헤더를 찾지 못함: {mer_file}")
            return

        df = pd.read_csv(
            mer_file,
            sep=";",
            skiprows=data_start_idx,
            encoding="utf-8",
            engine="python"
        )

        df.columns = [str(col).strip() for col in df.columns]

        parquet_path = mer_file.replace(".mer", ".parquet")

        df.to_parquet(
            parquet_path,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        print(f"[저장 완료] {parquet_path}")

        if delete_original:
            os.remove(mer_file)
            print(f"[원본 삭제 완료] {mer_file}")

    except Exception as e:
        print(f"[변환 실패] {mer_file}")
        print(e)


csv_full_path  = os.path.join(csv_path, csv_file_name)

# 시나리오 파일 읽기
senario_df = pd.read_csv(csv_full_path, encoding="cp949")

Vissim = win32com.client.Dispatch("Vissim.Vissim")  # Vissim 인스턴스 생성

for index, row in senario_df.iterrows():

    # 교통량
    Q_main_in_vehph = row["base_main_in_vph"]

    # 램프 유출 비율
    Q1_ramp_out_vehph = row["reflow"]

    # 유고 지속시간
    incident_duration_min = row["incident_duration_min"]

    shutil.copy(network_full_path, copy_network_path)

    # XML 로드
    tree = ET.parse(copy_network_path)
    root = tree.getroot()


    # 본선, 유입램프 교통량
    for vehicle_input in root.findall(".//vehicleInput"):
        input_name = vehicle_input.get("name")
        vehicleInput = vehicle_input.find("timeIntVehVols")

        if vehicleInput is None:
            continue
        for vol in vehicleInput.findall("timeIntervalVehVolume"):
            if input_name == "main":
                vol.set("volume", str(Q_main_in_vehph))

    # 램프 유출 비율
    for routing_decision  in root.findall(".//vehicleRoutingDecisionStatic"):
        decision_name = routing_decision.get("name")
        decision_no = routing_decision.get("no")

        veh_routes = routing_decision.find("vehRoutSta")
        if veh_routes is None:
            continue
        if decision_name == "상류" :
            for route in veh_routes.findall("vehicleRouteStatic"):
                route_no = route.get("no")

                if route_no == "1":  # 직진
                    route.set("relFlow", f"2 0:0.9, 2 1800000:{round(1- Q1_ramp_out_vehph, 3)}")

                elif route_no == "2":  # 진출
                    route.set("relFlow", f"2 0:0.1, 2 1800000:{round(Q1_ramp_out_vehph, 3)}")


    # 유고 발생 위치 조정
    for rsa in root.findall(".//reducedSpeedArea"):
        name = rsa.get("name")
        if name == "ugo":
            # 유고 지속시간 설정
            rsa.set("timeFrom", str(acc_start_time))
            rsa.set("timeTo", str(int(acc_start_time + incident_duration_min * 60)))



    print(
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        f"입력값 | main={Q_main_in_vehph}, 직진비율={1-Q1_ramp_out_vehph}, 진출비율={Q1_ramp_out_vehph}, 유고 지속시간={acc_start_time + incident_duration_min * 60}"
    )


    tree.write(copy_network_path, encoding="utf-8", xml_declaration=True)

    # VISSIM 네트워크 로드
    Vissim.LoadNet(copy_network_path)
    Vissim.Graphics.CurrentNetworkWindow.SetAttValue('QuickMode', 1) # 퀵 모드 활성화

    print(
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        "VISSIM 시뮬레이션 실행 중..."
    )
    Vissim.Simulation.RunContinuous()
    print(
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        "VISSIM 시뮬레이션 실행 완료"
    )

Vissim.Simulation.Stop()
Vissim.Exit()
print("VISSIM 시뮬레이션 종료")
print("==============================================================================================================")



[2026-06-12 00:17:00] 입력값 | main=2000, 직진비율=0.9, 진출비율=0.1, 유고 지속시간=2400
[2026-06-12 00:17:01] VISSIM 시뮬레이션 실행 중...
[2026-06-12 00:18:22] VISSIM 시뮬레이션 실행 완료
[2026-06-12 00:18:22] 입력값 | main=2000, 직진비율=0.8, 진출비율=0.2, 유고 지속시간=2400
[2026-06-12 00:18:23] VISSIM 시뮬레이션 실행 중...
[2026-06-12 00:19:45] VISSIM 시뮬레이션 실행 완료
[2026-06-12 00:19:45] 입력값 | main=2000, 직진비율=0.7, 진출비율=0.3, 유고 지속시간=2400
[2026-06-12 00:19:46] VISSIM 시뮬레이션 실행 중...
[2026-06-12 00:20:49] VISSIM 시뮬레이션 실행 완료
[2026-06-12 00:20:49] 입력값 | main=2000, 직진비율=0.6, 진출비율=0.4, 유고 지속시간=2400
[2026-06-12 00:20:50] VISSIM 시뮬레이션 실행 중...
[2026-06-12 00:21:51] VISSIM 시뮬레이션 실행 완료
[2026-06-12 00:21:51] 입력값 | main=2000, 직진비율=0.5, 진출비율=0.5, 유고 지속시간=2400
[2026-06-12 00:21:52] VISSIM 시뮬레이션 실행 중...
[2026-06-12 00:22:39] VISSIM 시뮬레이션 실행 완료
[2026-06-12 00:22:39] 입력값 | main=2000, 직진비율=0.4, 진출비율=0.6, 유고 지속시간=2400
[2026-06-12 00:22:40] VISSIM 시뮬레이션 실행 중...
[2026-06-12 00:23:26] VISSIM 시뮬레이션 실행 완료
[2026-06-12 00:23:26] 입력값 | main=2000, 직진비율=0.30000000000000004,